# Tahap 02: Data Cleaning

Tahap ini berfokus pada prapemrosesan data mentah yang telah dikumpulkan. Proses ini meliputi penghapusan data duplikat, penanganan nilai kosong (*missing values*), standardisasi tipe data, serta pembersihan teks komentar dari *noise* (seperti URL, *mention*, *hashtag*, dan angka). 

**Catatan Khusus:** Karakter emoji sengaja dipertahankan pada tahap pembersihan teks ini karena akan digunakan sebagai fitur penting dalam analisis selanjutnya.

In [ ]:
# --- LIBRARIES ---
import pandas as pd
import re

# --- 1. DATA LOADING ---
# Membaca file CSV hasil scraping sebelumnya
df = pd.read_csv('combined_instagram_comments.csv')
print(f"Jumlah data sebelum cleaning: {df.shape}")

# --- 2. DEDUPLICATION & MISSING VALUES HANDLING ---
# Menghapus duplikasi baris data
df = df.drop_duplicates()
print(f"Setelah hapus duplikat: {df.shape}")

# Menghapus baris yang memiliki nilai kosong (NaN)
df = df.dropna()

# --- 3. TEXT PREPROCESSING FUNCTION ---
def clean_text(text: str) -> str:
    """
    Membersihkan teks dari URL, mention, hashtag, angka, dan spasi berlebih,
    namun secara spesifik tetap mempertahankan emoji.
    """
    if pd.isna(text):
        return ""
    
    text = str(text)
    text = re.sub(r'http\S+|www.\S+', '', text)  # Hapus URL
    text = re.sub(r'@\w+|#\w+', '', text)        # Hapus mention & hashtag
    text = re.sub(r'\d+', '', text)              # Hapus angka
    text = re.sub(r'\s+', ' ', text).strip()     # Hapus spasi berlebih
    
    return text

# --- 4. DATA TRANSFORMATION & CLEANING ---
# Terapkan fungsi pembersihan pada kolom teks utama
df['text'] = df['text'].apply(clean_text)

# Bersihkan kolom 'ownerUsername' (hanya simpan karakter alfanumerik, _, .)
df['ownerUsername'] = df['ownerUsername'].astype(str).apply(
    lambda x: re.sub(r'[^A-Za-z0-9_.]', '', x)
)

# Konversi dan pastikan kolom 'likesCount' berupa tipe data integer murni
df['likesCount'] = pd.to_numeric(df['likesCount'], errors='coerce').fillna(0).astype(int)

# Bersihkan kolom URL dari spasi putih atau karakter tersembunyi
df['postUrl'] = df['postUrl'].astype(str).apply(lambda x: x.strip())
df['commentUrl'] = df['commentUrl'].astype(str).apply(lambda x: x.strip())

# Bersihkan kolom 'source_file' (hanya simpan karakter alfanumerik, _, ., -)
df['source_file'] = df['source_file'].astype(str).apply(
    lambda x: re.sub(r'[^A-Za-z0-9_.-]', '', x)
)

# Standardisasi format timestamp ke datetime dan hapus data waktu yang invalid (NaT)
df['timestamp'] = pd.to_datetime(df['timestamp'], errors='coerce')
df = df.dropna(subset=['timestamp'])

# Filter akhir: Hapus baris di mana kolom teks menjadi benar-benar kosong setelah dibersihkan
df = df[df['text'].str.strip() != ""]

# --- 5. EXPORT CLEANED DATA ---
print(f"Jumlah data setelah cleaning: {df.shape}")

# Menyimpan hasil akhir ke file CSV baru (disesuaikan nama string agar sinkron dengan nama file)
df.to_csv('cleaned_data.csv', index=False)
print("✅ Data cleaned berhasil disimpan ke 'cleaned_data.csv' (emoji tetap dipertahankan)")